# Paper 001: AAD experiment commands

Run these Linux-command cells from top to bottom. They clone or update the
repository, check one real video, check the custom model, and start one
controlled experiment stage. Errors are printed directly by the failing command.

## 1. Setup repository, packages, and AAD

In [ ]:
%%bash
set -euo pipefail

if [ -d /content/phd_research/.git ]; then
    git -C /content/phd_research pull --ff-only
else
    git clone https://github.com/sanelehlabisa/phd_research.git /content/phd_research
fi

cd /content/phd_research/papers/001-journal-abnormal-activity-recognition/implementation
python -m pip install -q av==16.1.0 kagglehub torchmetrics==1.8.2
python -c 'import kagglehub; path = kagglehub.dataset_download("sanelehlabisa/abnormal-activities-dataset"); print("Path to dataset files:", path)'

## 2. Load and augment one real AAD video

This checks decoding and augmentation without writing preview videos.

In [ ]:
%%bash
set -euo pipefail

cd /content/phd_research/papers/001-journal-abnormal-activity-recognition/implementation
DATASET_DIR="$(find /root/.cache/kagglehub/datasets/sanelehlabisa/abnormal-activities-dataset/versions -mindepth 2 -maxdepth 2 -type d -name abnormal-activities-dataset | sort -V | tail -n 1)"
test -n "$DATASET_DIR"
echo "Dataset: $DATASET_DIR"
python -c 'import sys; from src.dataset import AHARDataset, VideoAugmentation; dataset = AHARDataset(sys.argv[1], sequence_length=16, frame_size=(32, 32)); assert dataset.samples; video, label = dataset[0]; augmented = VideoAugmentation()(video.clone()); print("Samples:", len(dataset)); print("Classes:", dataset.class_names); print("Class:", dataset.class_names[label]); print("Clean:", tuple(video.shape), float(video.min()), float(video.max())); print("Augmented:", tuple(augmented.shape), float(augmented.min()), float(augmented.max()))' "$DATASET_DIR"

## 3. Run one clip through the model and list all candidates

In [ ]:
%%bash
set -euo pipefail

cd /content/phd_research/papers/001-journal-abnormal-activity-recognition/implementation
DATASET_DIR="$(find /root/.cache/kagglehub/datasets/sanelehlabisa/abnormal-activities-dataset/versions -mindepth 2 -maxdepth 2 -type d -name abnormal-activities-dataset | sort -V | tail -n 1)"
test -n "$DATASET_DIR"
python -c 'import sys, torch; from src.dataset import AHARDataset; from src.model import CustomConvLSTM, count_trainable_parameters; dataset = AHARDataset(sys.argv[1], sequence_length=16, frame_size=(32, 32)); video, _ = dataset[0]; device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); model = CustomConvLSTM(dataset.num_classes, layers=[(8, (3, 3)), (16, (3, 3))]).to(device).eval(); logits = model(video.unsqueeze(0).to(device)); print("Device:", device); print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"); print(model); print("Trainable parameters:", count_trainable_parameters(model)); print("Output shape:", tuple(logits.shape))' "$DATASET_DIR"
python -u -m src.experiments \
    --config configs/aad_screening_reference.json \
    --candidates-config configs/aad_architecture_candidates.json \
    --list-models

## 4. Run the architecture screen

Run this only after the two checks above pass. Change STAGE later for
baseline-confirmation, published-topology, or focused-ablations.

In [ ]:
%%bash
set -euo pipefail

cd /content/phd_research/papers/001-journal-abnormal-activity-recognition/implementation
DATASET_DIR="$(find /root/.cache/kagglehub/datasets/sanelehlabisa/abnormal-activities-dataset/versions -mindepth 2 -maxdepth 2 -type d -name abnormal-activities-dataset | sort -V | tail -n 1)"
test -n "$DATASET_DIR"
STAGE="architecture-screen"

python -u -m src.experiments \
    --plan-config configs/aad_controlled_experiment_plan.json \
    --run-plan-stage "$STAGE" \
    --plan-dataset-dir "$DATASET_DIR" \
    --plan-runs-dir runs

Results are written under implementation/runs/. The experiment command uses
validation results for selection and does not run final test evaluation.